# Stage 2 — OCR quality and preprocessing

**Goal.** Measure OCR quality against the reference text (CER, WER, share of years read correctly), clean the OCR text, remove PII and mask phrases that state the answer directly (total experience, age, date of birth), and create the shared train / validation / test split.

Logic: `src/ocr_quality.py`, `src/preprocessing.py`, `src/split.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import utils

utils.set_seed()
print("SEED:", utils.SEED, "| device:", utils.get_device(), "| root:", utils.ROOT)

The logic lives in `src/`; this notebook reads its outputs. Regenerate them with:

```bash
python -m src.ocr_quality      # CER / WER / years, error analysis
python -m src.preprocessing    # confidence analysis, cleaning, masking, data/processed/*.csv
python -m src.split            # data/splits/{train,val,test}.csv
```

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from src.ocr_extraction import load_ocr, ocr_text
from src.preprocessing import LOW_CONF_THRESHOLD, mask_direct_answers
from src.visualization import plot_ocr_quality

pd.set_option("display.max_colwidth", 120)

# Part 1 · OCR quality

**Normalisation.** OCR and reference text go through the same `normalize_text`: lowercase, NFKC, all unicode dashes to `-`, curly quotes to straight, bullet markers at line starts removed, single spaces around `|`, and all whitespace collapsed. Bullets are dropped because the OCR reads `-`/`•` inconsistently and they carry no content. Other punctuation is kept, because it is part of dates (`2019.11-2023.01`) and emails.

**Metrics** (`src/ocr_quality.py`):
- **CER** and **WER** per CV with `jiwer`, comparing the reading-order OCR text with the reference.
- **Years read correctly:** the share of four-digit years (1900–2099) in the reference that also appear in the OCR text, counting repeats. It is reported twice: on the OCR as is, and after `repair_date_ranges`.

In [ ]:
per_cv = pd.read_csv(utils.TABLES_DIR / "ocr_quality_per_cv.csv", dtype={"id": str})
summary = pd.read_csv(utils.TABLES_DIR / "ocr_quality_by_occupation.csv")
summary

**Why these charts.** Box plots show the whole per-CV distribution of CER and WER for each occupation (median, spread and outliers), which a single mean would hide. The bar chart puts the year recall before and after the date repair side by side, because years are the evidence the models need.

**What they show.** OCR errors are tiny (mean CER 0.21%, WER 1.9%). Software developers and nurses have slightly more errors (short uppercase tokens such as `API`, `CI`, `UI`, and more job lines). Only about 80% of years are found as is, and the repair brings that to about 100%.

In [ ]:
fig = plot_ocr_quality(per_cv, summary)
plt.show()

## Where the OCR fails

Each reference line is matched to its most similar OCR line, and the character errors are summed by line type. Almost all errors sit on the **job title + dates** lines: the OCR drops the en dash between the two dates (`2020.11–2023.01` → `2020.112023.01`). The digits are correct, but the second year is glued to the first and a year regex no longer sees it. That is why the year recall is about 80% as is, while CER is only 0.2%.

In [ ]:
pd.read_csv(utils.TABLES_DIR / "ocr_errors_by_line_type.csv")

Most frequent word-level errors (digits replaced by `#` to group them). Apart from the glued date range, the remaining errors are `i`→`l` confusions in short acronyms (`api`→`apl`, `ci/cd`→`cl/cd`, `ui`→`ul`) and a few truncated dates (`####.##-####.`).

The truncated dates are **not** OCR errors: for long job titles (mostly teachers), the PopResume image itself cuts the line before the month (`… Academy, 2020.02–2023.`), so the year is still read but the month is lost.

In [ ]:
pd.read_csv(utils.TABLES_DIR / "ocr_top_word_errors.csv").head(12)

# Part 2 · Reading 50 OCR texts: direct answers

`output/tables/sample_50_ocr_texts.txt` holds 50 random OCR texts (seed 42), in reading order and before cleaning. All 50 follow the same template: name, contact line (state, phone, email), `WORK HISTORY` (job title, employer, `YYYY.MM–YYYY.MM` range, 2 bullets per job) and `EDUCATION` (degree, institution, graduation year, bullets).

**None of them states total experience, age or date of birth.** A search over all 4,000 OCR texts and all 4,000 reference texts confirms it: no `N years (of) experience`, `age`, `years old`, `born` or `DOB`. The only matches for "experience" are the bullet *Led patient experience improvement programs*, which is not an answer.

The masking rule is applied anyway, so the pipeline is safe if other CVs are used. Below it is checked on synthetic sentences, with the guarantee that job dates and graduation years are not touched.

In [ ]:
print(open(utils.TABLES_DIR / "sample_50_ocr_texts.txt").read()[:1500])

In [ ]:
examples = [
    "Registered nurse with 12 years of experience in ICU care",
    "Over a decade of experience in auditing",
    "Experience: 8 years",
    "Age: 34",
    "John, 52 years old",
    "Date of birth: 03/12/1980",
    "Born in 1988 in Ohio",
    # must NOT be masked
    "Software Developers, Skywave Tech, 2019.11-2023.01",
    "Bachelor's, Ridgeway University, 2004",
    "- Led patient experience improvement programs",
]
pd.DataFrame({"text": examples, "after masking": [mask_direct_answers(t)[0] for t in examples]})

# Part 3 · Preprocessing

`src/preprocessing.py` builds the model inputs from the cached OCR, in this order:

1. **Reading order.** The lines come from `ocr_extraction.reading_order` (all CVs are single-column).
2. **Date-range repair.** Re-insert the dash between two glued `YYYY.MM` dates. It only fires on two valid dates (month 01–12) with no separator.
3. **Symbol normalisation.** Unicode dashes become `-` and bullet markers become `- `.
4. **Broken lines and hyphenation.** A line ending in `letter-` followed by a lowercase line is merged without the hyphen. A line starting in lowercase continues the previous line unless that one ends with `.`, `:`, `;`, `,` or `|`. PaddleOCR returns one box per visual line, so this rarely fires here; it is kept for robustness.
5. **PII removal.**
   - The name is the first line when the contact line follows it. It is detected on the full CV, so it is removed even if other lines are dropped.
   - The email, phone and `ST, USA` address are removed with regexes.
   - Names and location are forbidden features, so this applies to every variant.
6. **Masking of direct answers.** Total experience, age and date of birth become `[MASKED]`. It applies to every variant and affected **0 CVs**.

Job dates and graduation years are kept untouched: they are the evidence the models must reason from. `exp_year` is clipped to [0, 20] in every file.

In [ ]:
report = json.loads((utils.METRICS_DIR / "preprocessing_report.json").read_text())
report

### One CV before and after cleaning

A CV with glued date ranges: the raw OCR (left) and the `clean` text (right). The name, state, phone and email are gone, the date ranges have their dash back, and the bullets are uniform.

In [ ]:
log = pd.read_csv(utils.TABLES_DIR / "preprocessing_log.csv", dtype={"id": str})
cv_id = log.loc[log["glued_date_ranges"] >= 3, "id"].iloc[0]
clean = pd.read_csv(utils.PROCESSED_DIR / "clean.csv", dtype={"id": str}).set_index("id")
before = ocr_text(load_ocr(cv_id)).split("\n")
after = clean.loc[cv_id, "text"].split("\n")
n = max(len(before), len(after))
pd.DataFrame({"raw OCR": before + [""] * (n - len(before)), "clean": after + [""] * (n - len(after))})

## Low-confidence lines: is there a useful threshold?

**Why this chart.** The left histogram shows how confident PaddleOCR is. The right bars check whether low confidence actually means a wrong read: for each confidence bin, the share of lines whose text appears verbatim in the reference.

**What it shows.** Confidence is uniformly high: the minimum is 0.929, no line is below 0.9, and the mean is 0.991. It also does **not** predict errors: the lowest-confidence lines are read correctly 99–100% of the time. They are mostly names and short bullets, rare words for the recogniser. The least exact bins are the high-confidence date lines with the dropped dash, which the repair fixes anyway.

**Decision.**
- **No line is dropped in the main `clean` variant.** Any threshold would only remove correct text.
- For the `clean_dropped_lowconf` variant needed by Question 3, the threshold is **0.97**, about mean − 2.5 sd of the line scores, a standard outlier cut. It drops 2.1% of the lines (1,227 lines in 1,059 CVs). They are mostly bullets and names, but also 36 job-date lines. Question 3 will measure whether that costs anything.

In [ ]:
display(pd.read_csv(utils.TABLES_DIR / "ocr_confidence_by_bin.csv"))
display(pd.read_csv(utils.TABLES_DIR / "ocr_lowconf_lines_by_type.csv"))
fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(plt.imread(utils.FIGURES_DIR / "ocr_confidence.png"))
ax.axis("off")
ax.grid(False)
plt.show()

## Text variants

All four files have the same columns (`id, text, exp_year, occupation`) and the same 4,000 CVs:

| File | Content | Used for |
|---|---|---|
| `raw.csv` | reading-order OCR, PII removed and masked, no repair | Question 3 (raw vs cleaned) |
| `clean.csv` | + date repair, normalisation, line joining | **main model input** |
| `clean_dropped_lowconf.csv` | clean, without lines below 0.97 confidence | Question 3 |
| `reference_clean.csv` | the reference text through the same clean pipeline | **evaluation only** (Question 2), never a model input |

In [ ]:
variants = {name: pd.read_csv(utils.PROCESSED_DIR / f"{name}.csv", dtype={"id": str})
            for name in ["raw", "clean", "clean_dropped_lowconf", "reference_clean"]}
pd.DataFrame({name: {"CVs": len(df), "mean characters": df["text"].str.len().mean().round(1),
                     "mean lines": df["text"].str.count("\n").add(1).mean().round(2)}
              for name, df in variants.items()}).T

# Part 4 · Train / validation / test split

`src/split.py` splits by CV ID: 70% / 15% / 15%, stratified by **occupation × experience band** (20 strata), seed 42. Two chained `train_test_split` calls give 2,800 / 600 / 600 CVs. Every model reads `data/splits/{train,val,test}.csv`. Tuning uses validation only, and the test set is used once, in the final evaluation.

The three splits have the same mean target (≈ 6.5 years), the same band shares and the same occupation counts (± 1).

In [ ]:
pd.read_csv(utils.TABLES_DIR / "split_distribution.csv").set_index("split").T